# Run vs Submit

## What you'll learn

- Execute steps with `run()` (blocking) and `submit()` (non-blocking)
- Wire downstream steps before upstream ones finish using `StepFuture`
- Understand how step caching avoids redundant computation
- Resume a pipeline from where it left off

**Prerequisites:** [Your First Pipeline](01-first-pipeline.ipynb).
**Time:** ~10 min.
**GPU required:** No.

In [ ]:
from __future__ import annotations

import time

from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import Backend, PipelineManager
from artisan.utils import tutorial_setup
from artisan.utils.logging import configure_logging

configure_logging()

In [ ]:
env = tutorial_setup("run_vs_submit")
DELTA_ROOT = env.delta_root
STAGING_ROOT = env.staging_root
WORKING_ROOT = env.working_root

`PipelineManager` has two ways to execute a step: `run()` and `submit()`.
They accept the same arguments and produce the same results — the difference
is *when* they return.

| | `run()` | `submit()` |
|---|---|---|
| **Blocks?** | Yes — waits for the step to finish | No — returns immediately |
| **Returns** | `StepResult` | `StepFuture` |
| **Get outputs** | `result.output(role)` | `future.output(role)` |
| **Relationship** | `run()` = `submit().result()` | Underlying primitive |

Under the hood, `run()` calls `submit()` and waits for the result.
Everything else — caching, provenance, input wiring — is identical.

## Blocking execution with `run()`

`run()` blocks until the step finishes and returns a `StepResult` directly.
This is the default pattern you've already used in [Your First Pipeline](01-first-pipeline.ipynb).

In [ ]:
pipeline = PipelineManager.create(
    name="blocking",
    delta_root=DELTA_ROOT,
    staging_root=STAGING_ROOT,
    working_root=WORKING_ROOT,
)
output = pipeline.output

t0 = time.perf_counter()
step0 = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 3, "seed": 42},
    backend=Backend.LOCAL,
)
elapsed = time.perf_counter() - t0

print(f"Type:         {type(step0).__name__}")
print(f"Step name:    {step0.step_name}")
print(f"Success:      {step0.success}")
print(f"Output roles: {step0.output_roles}")
print(f"Wall time:    {elapsed:.3f}s (blocked until done)")

Because `run()` returns a `StepResult`, you can immediately use
`result.output(role)` to wire outputs into the next step.

In [ ]:
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 1, "seed": 100},
    backend=Backend.LOCAL,
)

pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    backend=Backend.LOCAL,
)

summary = pipeline.finalize()
print(
    f"\nPipeline: {summary['total_steps']} steps, success={summary['overall_success']}"
)

With `run()`, each step finishes before the next one starts. The pipeline
executes strictly top-to-bottom. This is the right choice when your steps are
fast or when you want to inspect intermediate results before continuing.

## Non-blocking execution with `submit()`

`submit()` returns a `StepFuture` immediately — the step runs in a
background thread. This lets you define your entire pipeline upfront
without waiting for each step to finish.

In [ ]:
env2 = tutorial_setup("run_vs_submit_async")

pipeline = PipelineManager.create(
    name="non_blocking",
    delta_root=env2.delta_root,
    staging_root=env2.staging_root,
    working_root=env2.working_root,
)
output = pipeline.output

In [ ]:
t0 = time.perf_counter()
future0 = pipeline.submit(
    operation=DataGenerator,
    name="generate",
    params={"count": 3, "seed": 42},
    backend=Backend.LOCAL,
)
elapsed = time.perf_counter() - t0

print(f"Type:    {type(future0).__name__}")
print(f"Status:  {future0.status}")
print(f"Done:    {future0.done}")
print(f"Elapsed: {elapsed:.4f}s (returned without waiting)")

### Wiring outputs before completion

The key advantage of `submit()`: you can call `future.output(role)` to get
an `OutputReference` *before* the step finishes. This reference is a lazy
placeholder — the framework resolves it to concrete artifact IDs when the
downstream step actually needs the data.

In [ ]:
ref = future0.output("datasets")
print(f"OutputReference: step={ref.source_step}, role={ref.role}")
print("(created instantly — step may still be running)")

This means you can wire your entire pipeline in one block. The framework
handles execution ordering — a downstream step automatically waits for
its upstream dependencies before dispatching.

In [ ]:
pipeline.submit(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 1, "seed": 100},
    backend=Backend.LOCAL,
)

future2 = pipeline.submit(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    backend=Backend.LOCAL,
)

# finalize() waits for all steps to complete
summary = pipeline.finalize()
print(
    f"\nPipeline: {summary['total_steps']} steps, success={summary['overall_success']}"
)

### Getting results from futures

If you need to inspect results after the pipeline runs, call
`future.result()` to get the underlying `StepResult`. After `finalize()`,
all futures are already complete, so this returns immediately.

In [ ]:
result = future2.result()
print(f"Step name:       {result.step_name}")
print(f"Succeeded count: {result.succeeded_count}")
print(f"Output roles:    {result.output_roles}")

### When to use which

| Situation | Use |
|---|---|
| Interactive exploration, debugging | `run()` — see each step's output before continuing |
| Production pipelines, SLURM submission | `submit()` — define the whole DAG upfront |
| Need to inspect intermediate results mid-pipeline | `run()` for that step |
| Steps with no data dependencies (parallel branches) | `submit()` — they can execute concurrently |

You can freely mix `run()` and `submit()` in the same pipeline.

## Step caching

Every step gets a deterministic `step_spec_id` computed from its operation
name, step number, parameters, and upstream spec IDs. When you run the same
pipeline with the same parameters, the framework detects the match and skips
execution entirely — the cached results are reused.

Let's re-run the exact same pipeline from the first section and observe
the cache hits.

In [ ]:
pipeline = PipelineManager.create(
    name="cache_demo",
    delta_root=DELTA_ROOT,
    staging_root=STAGING_ROOT,
    working_root=WORKING_ROOT,
)
output = pipeline.output

# Same operations, same params as the first section
t0 = time.perf_counter()
pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 3, "seed": 42},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 1, "seed": 100},
    backend=Backend.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    backend=Backend.LOCAL,
)
elapsed = time.perf_counter() - t0

pipeline.finalize()
print(f"\n3 steps completed in {elapsed:.3f}s (all cache hits — no re-execution)")

Notice the "CACHED — skipping execution" log messages and the fast completion
time. The framework found matching `step_spec_id`s in the Delta Lake tables
and reused the prior results.

Caching is **compositional** — the spec ID of each step depends on its
upstream spec IDs, not on artifact IDs. If the entire upstream chain
is identical, downstream steps cache-hit too.

Cache invalidation happens automatically when you change any of:
- Operation parameters
- Upstream inputs (different source data or different upstream params)
- Step position in the pipeline

## Resuming a pipeline

Pipeline state is persisted to Delta Lake. You can list past runs and resume
any of them — picking up from the last completed step.

In [ ]:
PipelineManager.list_runs(DELTA_ROOT)

In [ ]:
resumed = PipelineManager.resume(
    delta_root=DELTA_ROOT,
    staging_root=STAGING_ROOT,
)

print(f"Resumed run:  {resumed.config.pipeline_run_id}")
print(f"Current step: {resumed.current_step}")
print(f"Steps loaded: {len(resumed)}")

A resumed pipeline works like any other — you can access results from
prior steps and continue adding new ones.

In [ ]:
# Access results from prior steps using indexing
prior_result = resumed[2]
print(f"Step 2: {prior_result.step_name}, success={prior_result.success}")

# Wire prior outputs into new steps
ref = resumed[2].output("metrics")
print(f"Output ref: step={ref.source_step}, role={ref.role}")

## Summary

- **`run()`** blocks and returns a `StepResult` — use for interactive work
  and debugging
- **`submit()`** returns a `StepFuture` immediately — use for production
  pipelines and SLURM workflows where you define the full DAG upfront
- **Caching** skips re-execution when the same operation + params + inputs
  appear again, keyed by a deterministic `step_spec_id`
- **Resume** reconstructs pipeline state from Delta Lake, letting you
  continue from the last completed step

## Next steps

- [SLURM Execution](04-slurm-execution.ipynb) — Run steps on HPC clusters with `Backend.SLURM`
- [Pipeline Patterns](../pipeline-patterns/01-sources-and-chains.ipynb) — Sources, branching, filtering, and more
- [Execution Flow](../../concepts/execution-flow.md) — How steps are dispatched, batched, and committed